In [1]:
# build_tidyLong.ipynb (Starter Skeleton)

# ------------------------------
# Cell 1: Imports and Configuration
# ------------------------------
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Set project paths
project_root = Path("..")
eval_dirs = list(project_root.glob("evals_final_*/"))
mapfile_path = project_root / "postproc_intermediate" / "mapfile.csv"


In [2]:
# ------------------------------
# Cell 2: Filename Parser
# ------------------------------
import re

def parse_filename(fname: str):
    """
    Parse LLaMA or GPT eval filename into:
      - extract ∈ {"manual","automatic","fullNote","unknown"}
      - prompt  ∈ {"zeroShot","zeroShotCoT","fewShotK","fewShot","unknown"}
      - adversarial ∈ {True, False}
    Rules:
      • "GS_Extraction" or "GSExtraction" -> manual
      • "AE_Extraction" -> automatic
      • "Note_" or "Note__" -> fullNote
      • Prompts:
          ZS_DA  -> zeroShot
          ZS_CoT -> zeroShotCoT
          FS, FS_2shot, dict_FS_1shot -> fewShot / fewShotK
      • Adversarial:
          NoExpandedAbbrev OR Note__  -> True
          ExpandedAbbrev (and not Note__) -> False
    """
    meta = {}
    f = fname.replace(".csv", "")

    # ---- Extraction ----
    if "GS_Extraction" in f or "GSExtraction" in f:
        meta["extract"] = "manual"
    elif "AE_Extraction" in f:
        meta["extract"] = "automatic"
    elif "Note__" in f or "Note_" in f:
        meta["extract"] = "fullNote"
    else:
        meta["extract"] = "unknown"

    # ---- Prompt ----
    if "ZS_CoT" in f:
        meta["prompt"] = "zeroShotCoT"
    elif "ZS_DA" in f:
        meta["prompt"] = "zeroShot"
    else:
        if "FS" in f:
            # Try to capture K in: FS_2shot, dict_FS_1shot, etc.
            m = re.search(r'(?:dict_)?FS[_-]?(\d)shot', f)
            if not m:
                # fallback: any "<digit>shot" when FS is present
                m = re.search(r'(\d)shot', f)
            meta["prompt"] = f"fewShot{m.group(1)}" if m else "fewShot"
        else:
            meta["prompt"] = "unknown"

    # ---- Adversarial ----
    # Adversarial if abbreviations were NOT expanded:
    #   • filenames that explicitly say "NoExpandedAbbrev"
    #   • NOTE runs using the double-underscore form "Note__..."
    is_noexp = ("NoExpandedAbbrev" in f) or ("Note__" in f)
    meta["adversarial"] = bool(is_noexp)

    return meta


In [3]:
# ------------------------------
# Cell 3: Load Mapfile
# ------------------------------
# Load and prepare mapfile
mapfile = pd.read_csv(mapfile_path)

# Standardize column names if needed
mapfile.rename(columns={"Sample_ID": "patient_id"}, inplace=True)

# Optional: validate expected columns
required_cols = ["patient_id", "gold_NIHSS_total", "silver_NIHSS_total", "fold"]
missing_cols = [col for col in required_cols if col not in mapfile.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in mapfile: {missing_cols}")



In [4]:
# ------------------------------
# Cell 4: Initialize Output List
# ------------------------------
# Store results from all models and files
all_results = []



In [5]:
# ------------------------------
# Cell 5: Loop Over Model Folders and Files
# ------------------------------
for model_dir in eval_dirs:
    model = model_dir.name.replace("evals_final_", "")  # e.g., 'llama', 'gpt'

    for file in model_dir.glob("*.csv"):
        meta = parse_filename(file.name)
        meta["model"] = model

        # Load prediction file and retain only relevant columns
        df = pd.read_csv(file)
        df = df.rename(columns={
            "Sample_ID": "patient_id",
            "NIHSS_pred_total": "pred_score"
        })[["patient_id", "pred_score"]]  # Drop extra columns

        # Ensure predictions are treated as integer labels
        df["pred_score"] = df["pred_score"].round().astype(int)

        # Add metadata columns to df
        for k, v in meta.items():
            df[k] = v

        # Merge with mapfile
        merged = df.merge(mapfile, on="patient_id", how="inner")

        # For each reference type: gold and silver
        for ref in ["gold", "silver"]:
            ref_col = f"{ref}_NIHSS_total"
            if ref_col not in merged.columns:
                continue

            subdf = merged.copy()
            subdf["reference"] = ref

            # Compute basic error metrics
            subdf["abs_error"] = (subdf["pred_score"] - subdf[ref_col]).abs()
            subdf["squared_error"] = (subdf["pred_score"] - subdf[ref_col]) ** 2

            # Append to master list
            all_results.append(subdf)

In [6]:
# ------------------------------
# Cell 6: Concatenate Final DataFrame
# ------------------------------
# Combine all rows into a single tidy DataFrame
final_df = pd.concat(all_results, ignore_index=True)


# Quick inspection
print(f"Final shape: {final_df.shape}")
print("Columns:", final_df.columns.tolist())
print("Models:", final_df["model"].unique())
print("References:", final_df["reference"].unique())

Final shape: (106094, 46)
Columns: ['patient_id', 'pred_score', 'extract', 'prompt', 'adversarial', 'model', 'gold_NIHSS_total', 'gold_1a', 'gold_1b', 'gold_1c', 'gold_2', 'gold_3', 'gold_4', 'gold_5a', 'gold_5b', 'gold_6a', 'gold_6b', 'gold_7', 'gold_8', 'gold_9', 'gold_10', 'gold_11', 'silver_NIHSS_total', 'silver_1a', 'silver_1b', 'silver_1c', 'silver_2', 'silver_3', 'silver_4', 'silver_5a', 'silver_5b', 'silver_6a', 'silver_6b', 'silver_7', 'silver_8', 'silver_9', 'silver_10', 'silver_11', 'admit_date', 'year_bin', 'source_silver', 'dataset_source_flag', 'fold', 'reference', 'abs_error', 'squared_error']
Models: ['gpt' 'llama']
References: ['gold' 'silver']


In [7]:
# ------------------------------
# Cell 7: Save Final Output Per Model
# ------------------------------
output_dir = project_root / "postproc_intermediate"

for model in final_df["model"].unique():
    df_model = final_df[final_df["model"] == model]
    outpath = output_dir / f"{model}_tidyLong.csv"
    df_model.to_csv(outpath, index=False)
    print(f"Saved: {outpath.name} ({df_model.shape[0]} rows)")

Saved: gpt_tidyLong.csv (21206 rows)
Saved: llama_tidyLong.csv (84888 rows)
